# 02 · Precios y recorte a ~1.000 tickers

**Objetivo:** descargar con `yfinance` los precios diarios (OHLCV) del universo NASDAQ + NYSE y quedarnos con las ~1.000 acciones más líquidas. Se hace en dos filtros: primero un **suelo de capitalización** y después un ranking por **volumen mediano en dólares** (cierre × volumen). Los umbrales se eligen viendo la distribución real de los datos.

**Entradas:**
- `data/cache/universo_nasdaq_nyse.parquet` (salida del notebook 01)

**Salidas:**
- `data/cache/precios_recientes_universo.parquet` — OHLCV de ~6 meses de los 7.659 tickers (caché de la descarga)
- `data/cache/capitalizacion_candidatos.parquet` — tipo de valor y capitalización de los candidatos (caché de yfinance)
- `data/cache/universo_recortado.parquet` — los 1.000 tickers elegidos, con CIK, capitalización y liquidez
- `data/cache/precios_historicos.parquet` — OHLCV diario ajustado de 10 años de los 1.000 tickers elegidos (entrada de los notebooks 04 y 05)

In [1]:
import time

import pandas as pd
import plotly.express as px
import yfinance as yf
from dotenv import load_dotenv

from funciones import CARPETA_CACHE

load_dotenv();

## 1. Universo de partida

Leemos de disco la salida del notebook 01: los 7.659 tickers de NASDAQ + NYSE.

In [2]:
universo = pd.read_parquet(CARPETA_CACHE / "universo_nasdaq_nyse.parquet")
print("Tickers en el universo:", len(universo))
universo.head()

Tickers en el universo: 7659


,cik,ticker,name,exchange
0,0001045810,NVDA,NVIDIA CORP,Nasdaq
1,0000320193,AAPL,Apple Inc.,Nasdaq
2,0001652044,GOOGL,Alphabet Inc.,Nasdaq
3,0000789019,MSFT,MICROSOFT CORP,Nasdaq
4,0001018724,AMZN,AMAZON COM INC,Nasdaq


## 2. ¿Qué devuelve yfinance? (respuesta real con 3 tickers)

Antes de descargar 7.659 tickers miramos el formato real con una muestra pequeña (regla 4 del proyecto: no asumir el formato de los datos).

- `auto_adjust=False`: los precios vienen ajustados por *splits*, pero no por dividendos. Para calcular el volumen en dólares nos interesa el precio que realmente se negoció.
- Con varios tickers, yfinance devuelve columnas con **dos niveles**: el campo (`Open`, `Close`…) y el ticker.

> **Versión de yfinance:** la 0.2.43 dejó de funcionar porque Yahoo responde `429 Too Many Requests` a los clientes que no parecen un navegador. Se actualizó a la 1.7.0, que se identifica como navegador mediante `curl_cffi`. Detalle en la bitácora.

In [3]:
print("Versión de yfinance:", yf.__version__)

ejemplo = yf.download(
    ["AAPL", "BRK-B", "XFLH-RI"], start="2026-10-01", end="2026-10-10",
    auto_adjust=False, progress=False,
)
print("Niveles de columnas:", ejemplo.columns.names)
ejemplo

Versión de yfinance: 1.7.0


$XFLH-RI: possibly delisted; no price data found  (1d 2026-10-01 -> 2026-10-10)



1 Failed download:


['XFLH-RI']: possibly delisted; no price data found  (1d 2026-10-01 -> 2026-10-10)


Niveles de columnas: ['Price', 'Ticker']


Price        Adj Close                           Close                      \
Ticker            AAPL       BRK-B XFLH-RI        AAPL       BRK-B XFLH-RI   
Date                                                                         
2026-10-01  330.320007  500.500000     NaN  330.320007  500.500000     NaN   
2026-10-02  333.690002  502.649994     NaN  333.690002  502.649994     NaN   
2026-10-05  332.890015  504.260010     NaN  332.890015  504.260010     NaN   
2026-10-06  333.630005  505.540009     NaN  333.630005  505.540009     NaN   
2026-10-07  336.670013  506.250000     NaN  336.670013  506.250000     NaN   
2026-10-08  340.420013  511.049988     NaN  340.420013  511.049988     NaN   
2026-10-09  336.640015  515.619995     NaN  336.640015  515.619995     NaN   

Price             High                             Low                      \
Ticker            AAPL       BRK-B XFLH-RI        AAPL       BRK-B XFLH-RI   
Date                                                                         
2026-10-01  332.480011  501.529999     NaN  325.809998  495.959991     NaN   
2026-10-02  334.540009  503.619995     NaN  330.609985  499.010010     NaN   
2026-10-05  336.209991  505.880005     NaN  331.649994  501.299988     NaN   
2026-10-06  334.380005  508.519989     NaN  330.619995  503.779999     NaN   
2026-10-07  338.670013  506.700012     NaN  332.779999  504.299988     NaN   
2026-10-08  341.570007  513.150024     NaN  335.899994  505.269989     NaN   
2026-10-09  338.609985  516.570007     NaN  330.700012  509.709991     NaN   

Price             Open                        Volume                   
Ticker            AAPL       BRK-B XFLH-RI      AAPL    BRK-B XFLH-RI  
Date                                                                   
2026-10-01  330.000000  497.250000     NaN  36306300  4594800     NaN  
2026-10-02  333.260010  500.100006     NaN  33278600  4189700     NaN  
2026-10-05  332.820007  502.049988     NaN  34400900  3567500     NaN  
2026-10-06  332.279999  504.339996     NaN  30449000  3248000     NaN  
2026-10-07  336.959991  506.119995     NaN  34147900  3499200     NaN  
2026-10-08  336.820007  506.609985     NaN  35332400  4678900     NaN  
2026-10-09  331.700012  510.000000     NaN  37881200  4304300     NaN

Se observan dos cosas:

1. Los tickers con guion (p. ej. `BRK-B`) se aceptan tal cual: la SEC y Yahoo usan la misma notación, así que no hace falta traducirlos.
2. El universo incluye valores que **no son acciones ordinarias**, como `XFLH-RI` (un derecho de suscripción) o las acciones preferentes con sufijo `-P…`. Suelen ser muy poco líquidos, así que el ranking por volumen debería dejarlos fuera de forma natural. Se comprobará en la sección de recorte.

## 3. Descarga de ~6 meses de precios de todo el universo

Para ordenar por liquidez bastan unos meses: la ventana de liquidez será de **63 sesiones** (~3 meses de mercado) y descargamos algo más de margen. El histórico largo (10 años) solo se descargará después para los ~1.000 tickers elegidos, porque bajarlo para 7.659 sería mucho más lento y pesado sin necesidad.

Fechas **fijas** para que el resultado sea reproducible: del 1 abr 2026 al 9 oct 2026 (último viernes con mercado; en yfinance la fecha `end` no se incluye).

La descarga va en **lotes de 200 tickers** con una pausa de 1 s entre lotes, para no provocar bloqueos de Yahoo. El resultado se guarda en formato **largo** (una fila por ticker y día), más fácil de leer y filtrar que las columnas de dos niveles. Si el archivo de caché ya existe, se lee de disco y no se descarga nada.

In [4]:
FECHA_INICIO = "2026-04-01"
FECHA_FIN = "2026-10-10"   # no incluida: el último día descargado es el 9 oct
TAMANO_LOTE = 200
PAUSA_ENTRE_LOTES = 1.0


def descargar_precios_en_lotes(tickers: list[str], inicio: str, fin: str, ajustados: bool = False) -> pd.DataFrame:
    """Descarga OHLCV diario de yfinance por lotes y lo devuelve en formato largo.

    Recibe la lista de tickers, las fechas (fin no incluida) y si se quieren
    precios ajustados por splits y dividendos. Devuelve un
    DataFrame con columnas fecha, ticker, open, high, low, close, volume,
    sin las filas vacías (días en que un ticker no cotizó).
    """
    trozos = []
    for posicion in range(0, len(tickers), TAMANO_LOTE):
        lote = tickers[posicion : posicion + TAMANO_LOTE]
        datos = yf.download(lote, start=inicio, end=fin, auto_adjust=ajustados, progress=False)
        if datos.empty:
            raise RuntimeError(f"yfinance no devolvió datos para el lote que empieza en {lote[0]}")
        largo = datos.stack(level="Ticker", future_stack=True).reset_index()
        trozos.append(largo)
        print(f"  lote {posicion // TAMANO_LOTE + 1}: {len(lote)} tickers")
        time.sleep(PAUSA_ENTRE_LOTES)

    precios = pd.concat(trozos, ignore_index=True)
    precios = precios.rename(columns={"Date": "fecha", "Ticker": "ticker", "Open": "open", "High": "high",
                                      "Low": "low", "Close": "close", "Volume": "volume"})
    precios = precios[["fecha", "ticker", "open", "high", "low", "close", "volume"]]
    return precios.dropna(subset=["close"]).reset_index(drop=True)

In [5]:
ruta_precios_recientes = CARPETA_CACHE / "precios_recientes_universo.parquet"

if ruta_precios_recientes.exists():
    precios_recientes = pd.read_parquet(ruta_precios_recientes)
    print("Leído de caché:", ruta_precios_recientes.name)
else:
    inicio_descarga = time.time()
    precios_recientes = descargar_precios_en_lotes(universo["ticker"].tolist(), FECHA_INICIO, FECHA_FIN)
    precios_recientes.to_parquet(ruta_precios_recientes, index=False)
    print(f"Descargado en {time.time() - inicio_descarga:.0f} s y guardado en {ruta_precios_recientes.name}")

print("Filas:", len(precios_recientes))
print("Tickers con datos:", precios_recientes["ticker"].nunique(), "de", len(universo))
print("Fechas:", precios_recientes["fecha"].min().date(), "→", precios_recientes["fecha"].max().date())

Leído de caché: precios_recientes_universo.parquet
Filas: 919064
Tickers con datos: 7652 de 7659
Fechas: 2026-04-01 → 2026-10-09


### Tickers sin datos o con poco histórico

Algunos tickers de la SEC pueden no tener precios en Yahoo (deslistados recientemente, nombres distintos…) o haber empezado a cotizar hace poco. Los contamos antes de seguir, para no perderlos en silencio.

In [6]:
sesiones_por_ticker = precios_recientes.groupby("ticker").size()
tickers_sin_datos = sorted(set(universo["ticker"]) - set(sesiones_por_ticker.index))

print("Sin ningún dato:", len(tickers_sin_datos))
print("  ejemplos:", tickers_sin_datos[:15])
print("Con menos de 63 sesiones (no se puede calcular la liquidez de 63 sesiones):",
      (sesiones_por_ticker < 63).sum())

Sin ningún dato: 7
  ejemplos: ['ACHR-WT', 'FBYDP', 'IPXGU', 'PLTS', 'SVA', 'SY', 'UCFIW']
Con menos de 63 sesiones (no se puede calcular la liquidez de 63 sesiones): 727


¿Quiénes son los que tienen menos de 63 sesiones? Si fueran empresas recién salidas a bolsa, perderíamos acciones interesantes. Miramos cuántas sesiones tienen y una muestra con su nombre:

In [7]:
pocas_sesiones = sesiones_por_ticker[sesiones_por_ticker < 63]
print("Mediana de sesiones con datos:", pocas_sesiones.median(), "de ~133 posibles")

muestra_pocas = universo.set_index("ticker").loc[pocas_sesiones.sample(12, random_state=1).index, ["name"]]
muestra_pocas["sesiones"] = pocas_sesiones
muestra_pocas

Mediana de sesiones con datos: 1.0 de ~133 posibles


,name,sesiones
ticker,,
FTRA-WT,FutureCorp Space Acquisition 1,1
UAC-WT,United Acquisition Corp. I,1
CRACR,Crown Reserve Acquisition Corp. I,1
KITTW,"Nauticus Robotics, Inc.",1
PECER,Peace Acquisition Corp.,1
PAII-WT,Pyrophyte Acquisition Corp. II,1
PECEW,Peace Acquisition Corp.,1
MDCXW,Medicus Pharma Ltd.,1
NVNIW,Nvni Group Ltd,1


Casi todos son **warrants** (sufijos `W` o `-WT`), **derechos** (`R`) o **acciones preferentes** (`-P…`) que solo negocian algún día suelto: yfinance solo devuelve fila los días con operaciones. No son acciones ordinarias ni tienen liquidez, así que excluirlos no hace perder nada relevante.

## 4. Liquidez: volumen mediano en dólares (últimas 63 sesiones)

El **volumen en dólares** de un día es *cierre × volumen*: cuánto dinero cambió de manos. Su valor típico en las últimas 63 sesiones mide la liquidez: cuanto más alto, más fácil es comprar o vender la acción sin mover el precio.

Usamos la **mediana** y no la media. Un solo día excepcional (una noticia, una *meme stock*) dispara la media y puede colar en el ranking una empresa que normalmente apenas negocia; la mediana no se mueve por un día aislado. Calculamos también la media para compararlas en la sección de sensibilidad. Solo entran los tickers con las 63 sesiones completas.

In [8]:
VENTANA_VOLUMEN = 63

precios_recientes["volumen_dolares"] = precios_recientes["close"] * precios_recientes["volume"]
ultimas_sesiones = precios_recientes.sort_values("fecha").groupby("ticker").tail(VENTANA_VOLUMEN)

liquidez = ultimas_sesiones.groupby("ticker").agg(
    volumen_mediano_dolares=("volumen_dolares", "median"),
    volumen_medio_dolares=("volumen_dolares", "mean"),
    sesiones=("volumen_dolares", "size"),
)
liquidez = liquidez[liquidez["sesiones"] == VENTANA_VOLUMEN].sort_values("volumen_mediano_dolares", ascending=False)

print("Tickers con liquidez calculada:", len(liquidez))
liquidez.head(10)

Tickers con liquidez calculada: 6925


,volumen_mediano_dolares,volumen_medio_dolares,sesiones
ticker,,,
SPY,3.268734e+10,3.309001e+10,63
MU,2.938481e+10,3.055576e+10,63
NVDA,2.496771e+10,2.621104e+10,63
SNDK,1.691481e+10,1.917220e+10,63
AAPL,1.267465e+10,1.465376e+10,63
TSLA,1.179943e+10,1.327386e+10,63
AMD,1.162112e+10,1.211503e+10,63
MSFT,1.098514e+10,1.302542e+10,63
SPCX,1.076495e+10,1.313202e+10,63


### Distribución del volumen mediano en dólares

La liquidez está muy concentrada: unas pocas acciones mueven miles de millones al día y la mayoría, muy poco. Por eso el histograma usa **escala logarítmica** en el eje X (cada marca es 10 veces la anterior).

En la tabla de arriba aparecen **SPY** y **QQQ**: son ETF (fondos que replican un índice), no empresas. La SEC también lista algunos fondos cotizados. Se eliminan en la sección 5.

In [9]:
grafico = px.histogram(
    liquidez, x="volumen_mediano_dolares", log_x=True, nbins=80,
    labels={"volumen_mediano_dolares": "Volumen mediano diario en dólares (63 sesiones, escala log)"},
    title="Distribución de la liquidez del universo NASDAQ + NYSE",
)
grafico.show(renderer="notebook_connected")

In [10]:
print("Cuantiles del volumen mediano diario en dólares:")
display(liquidez["volumen_mediano_dolares"].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).map("{:,.0f} $".format))

for umbral in [1e6, 5e6, 10e6, 20e6, 50e6]:
    print(f"Tickers con más de {umbral / 1e6:>4.0f} M$/día: {(liquidez['volumen_mediano_dolares'] > umbral).sum():>5}")

Cuantiles del volumen mediano diario en dólares:


0.10         17,186 $
0.25        153,252 $
0.50      2,176,500 $
0.75     32,512,857 $
0.90    152,133,934 $
Name: volumen_mediano_dolares, dtype: object

Tickers con más de    1 M$/día:  3956
Tickers con más de    5 M$/día:  2954
Tickers con más de   10 M$/día:  2523
Tickers con más de   20 M$/día:  2072
Tickers con más de   50 M$/día:  1422


## 5. Recorte a 1.000 tickers

El recorte tiene tres pasos:

1. **Prefiltro de volumen (solo por coste):** pedir la capitalización a yfinance cuesta ~0,3 s por ticker. Solo se pide a los que tienen un volumen mediano ≥ 10 M$/día. Este prefiltro **no cambia el resultado** siempre que el ticker nº 1.000 elegido quede por encima de 10 M$; se verifica en Comprobaciones.
2. **Solo acciones ordinarias, con un suelo de capitalización de 2.000 M$.** Es la frontera habitual entre *small caps* y *mid caps*. El suelo protege frente a empresas pequeñas con picos puntuales de negociación.
3. **Ranking por volumen mediano en dólares:** nos quedamos con las 1.000 primeras.

In [11]:
PREFILTRO_VOLUMEN = 10e6
SUELO_CAPITALIZACION = 2e9
NUM_TICKERS = 1_000

candidatos = liquidez[liquidez["volumen_mediano_dolares"] >= PREFILTRO_VOLUMEN].index.tolist()
print("Candidatos tras el prefiltro de volumen:", len(candidatos))

Candidatos tras el prefiltro de volumen: 2523


### ¿Qué devuelve yfinance sobre el tipo de valor y la capitalización?

Usamos `fast_info`, la consulta rápida de yfinance. Miramos la respuesta real para una acción (AAPL), un ETF (SPY), una acción con guion (BRK-B), una preferente (MER-PK) y un warrant (KITTW):

In [12]:
for ticker in ["AAPL", "SPY", "BRK-B", "MER-PK", "KITTW"]:
    info = yf.Ticker(ticker).fast_info
    print(f"{ticker:7} tipo={info['quoteType']:7} capitalización={info['marketCap']}")

AAPL    tipo=EQUITY  capitalización=5004928124830.322


SPY     tipo=ETF     capitalización=None


BRK-B   tipo=EQUITY  capitalización=1139349931849.8818


MER-PK  tipo=EQUITY  capitalización=None


KITTW   tipo=EQUITY  capitalización=None


- Los **ETF** vienen con `quoteType = 'ETF'` y sin capitalización.
- Las **preferentes y los warrants** vienen como `'EQUITY'`, pero **sin capitalización**: Yahoo no la calcula para ellos.

Por eso, exigir `EQUITY` **y** una capitalización por encima del suelo elimina las tres cosas a la vez.

La consulta se guarda en caché **cada 250 tickers**. Si la descarga se corta, al volver a ejecutar se continúa donde se quedó. Un error en un ticker no detiene el proceso: se anota en la columna `error` y se cuenta al final.

In [13]:
def pedir_tipo_y_capitalizacion(ticker: str) -> dict:
    """Pide a yfinance el tipo de valor y la capitalización de un ticker.

    Devuelve un diccionario con ticker, tipo, capitalizacion y error.
    Si la consulta falla, tipo y capitalizacion quedan vacíos y el
    mensaje se guarda en error, para contarlo después en lugar de ocultarlo.
    """
    try:
        info = yf.Ticker(ticker).fast_info
        return {"ticker": ticker, "tipo": info["quoteType"], "capitalizacion": info["marketCap"], "error": None}
    except Exception as error:  # yfinance lanza errores de tipos muy variados
        return {"ticker": ticker, "tipo": None, "capitalizacion": None, "error": str(error)}

In [14]:
ruta_capitalizacion = CARPETA_CACHE / "capitalizacion_candidatos.parquet"

filas = pd.read_parquet(ruta_capitalizacion).to_dict("records") if ruta_capitalizacion.exists() else []
ya_pedidos = {fila["ticker"] for fila in filas}
pendientes = [ticker for ticker in candidatos if ticker not in ya_pedidos]
print("Ya en caché:", len(ya_pedidos), "· Pendientes de pedir:", len(pendientes))

inicio_consulta = time.time()
for numero, ticker in enumerate(pendientes, start=1):
    filas.append(pedir_tipo_y_capitalizacion(ticker))
    if numero % 250 == 0 or numero == len(pendientes):
        pd.DataFrame(filas).to_parquet(ruta_capitalizacion, index=False)
        print(f"  {numero}/{len(pendientes)} ({time.time() - inicio_consulta:.0f} s)")

capitalizacion = pd.DataFrame(filas)
capitalizacion = capitalizacion[capitalizacion["ticker"].isin(candidatos)].set_index("ticker")
print("Errores en la consulta:", capitalizacion["error"].notna().sum())

Ya en caché: 2523 · Pendientes de pedir: 0
Errores en la consulta: 0


In [15]:
print("Tipos de valor entre los candidatos:")
display(capitalizacion["tipo"].value_counts(dropna=False))

sin_capitalizacion = capitalizacion["capitalizacion"].isna()
print("Candidatos sin capitalización, por tipo:")
capitalizacion.loc[sin_capitalizacion, "tipo"].value_counts(dropna=False)

Tipos de valor entre los candidatos:


tipo
EQUITY    2468
ETF         55
Name: count, dtype: int64

Candidatos sin capitalización, por tipo:


tipo
ETF       55
EQUITY    11
Name: count, dtype: int64

### Distribución de la capitalización de las acciones candidatas

Las líneas verticales marcan las fronteras convencionales de tamaño: 300 M$ (*micro / small*), 2.000 M$ (*small / mid*, el suelo elegido) y 10.000 M$ (*mid / large*).

In [16]:
acciones = capitalizacion[(capitalizacion["tipo"] == "EQUITY") & capitalizacion["capitalizacion"].notna()]
acciones = acciones.join(liquidez[["volumen_mediano_dolares", "volumen_medio_dolares"]])
acciones = acciones.join(universo.set_index("ticker")["cik"])
print("Acciones ordinarias con capitalización:", len(acciones))

grafico = px.histogram(
    acciones, x="capitalizacion", log_x=True, nbins=80,
    labels={"capitalizacion": "Capitalización (dólares, escala log)"},
    title="Capitalización de las acciones candidatas",
)
for frontera in [300e6, 2e9, 10e9]:
    grafico.add_vline(x=frontera, line_dash="dash")
grafico.show(renderer="notebook_connected")

Acciones ordinarias con capitalización: 2457


### Selección

Aplicamos el suelo de capitalización y nos quedamos con las 1.000 de mayor volumen mediano. La misma función se reutiliza en la sección de sensibilidad con otros suelos y con la media.

**Una empresa, un ticker.** Algunas empresas cotizan con dos clases de acciones bajo el mismo CIK: Alphabet (`GOOGL` y `GOOG`), Berkshire Hathaway (`BRK-B` y `BRK-A`) y Petrobras (`PBR` y `PBR-A`). Comparten filings y se mueven casi igual, así que contarlas dos veces duplicaría la empresa en el backtesting y en el sentimiento. Se queda la clase **más líquida** de cada CIK y se completa hasta 1.000 con las siguientes del ranking.

In [17]:
def seleccionar_tickers(suelo: float, columna_volumen: str = "volumen_mediano_dolares") -> pd.Index:
    """Devuelve los NUM_TICKERS tickers más líquidos con capitalización >= suelo.

    Recibe el suelo en dólares y la columna de liquidez para ordenar
    (mediana por defecto). Lanza ValueError si no hay suficientes acciones.
    """
    elegibles = acciones[acciones["capitalizacion"] >= suelo].sort_values(columna_volumen, ascending=False)
    # Una sola clase de acción por empresa: como están ordenadas por liquidez, se queda la primera de cada CIK
    elegibles = elegibles[~elegibles["cik"].duplicated()]
    if len(elegibles) < NUM_TICKERS:
        raise ValueError(f"Con suelo {suelo:,.0f} $ solo hay {len(elegibles)} empresas elegibles")
    return elegibles.head(NUM_TICKERS).index


seleccion = seleccionar_tickers(SUELO_CAPITALIZACION)
elegidos = acciones.loc[seleccion]

print("Acciones elegibles con el suelo de 2.000 M$:", (acciones["capitalizacion"] >= SUELO_CAPITALIZACION).sum())
print("Seleccionadas:", len(elegidos))
print(f"Volumen mediano del nº 1.000: {elegidos['volumen_mediano_dolares'].min() / 1e6:,.1f} M$/día")
print(f"Capitalización mínima seleccionada: {elegidos['capitalizacion'].min() / 1e9:,.2f} mil M$")

Acciones elegibles con el suelo de 2.000 M$: 1945
Seleccionadas: 1000
Volumen mediano del nº 1.000: 84.4 M$/día
Capitalización mínima seleccionada: 2.13 mil M$


Qué clases de acción quedan fuera por compartir CIK con otra más líquida:

In [18]:
con_cik_repetido = acciones[(acciones["capitalizacion"] >= SUELO_CAPITALIZACION) & acciones["cik"].duplicated(keep=False)]
con_cik_repetido = con_cik_repetido.join(universo.set_index("ticker")["name"]).sort_values(["cik", "volumen_mediano_dolares"], ascending=[True, False])
con_cik_repetido["elegida"] = con_cik_repetido.index.isin(seleccion)
con_cik_repetido[["cik", "name", "volumen_mediano_dolares", "elegida"]]

,cik,name,volumen_mediano_dolares,elegida
ticker,,,,
HEI,0000046619,HEICO CORP,1.451035e+08,True
HEI-A,0000046619,HEICO CORP,6.141623e+07,False
BELFB,0000729580,BEL FUSE INC /NJ,4.046616e+07,False
BELFA,0000729580,BEL FUSE INC /NJ,2.569577e+07,False
BRK-B,0001067983,BERKSHIRE HATHAWAY INC,2.066803e+09,True
BRK-A,0001067983,BERKSHIRE HATHAWAY INC,1.506736e+08,False
PBR,0001119639,PETROBRAS - PETROLEO BRASILEIRO SA,3.479608e+08,True
PBR-A,0001119639,PETROBRAS - PETROLEO BRASILEIRO SA,1.234534e+08,False
UAA,0001336917,"Under Armour, Inc.",4.684340e+07,False


## 6. Sensibilidad: ¿cuánto depende la selección de las decisiones tomadas?

Se repite la selección cambiando **una sola decisión cada vez** y se cuenta cuántos de los 1.000 tickers cambian respecto a la selección elegida (suelo de 2.000 M$ y mediana). Si cambian pocos, el valor exacto del umbral apenas influye y la elección queda justificada con datos, no por intuición. Una variante sin 1.000 acciones elegibles se marca como no viable (celdas vacías).

In [19]:
variantes = {
    "Sin suelo (solo acciones con capitalización)": (0, "volumen_mediano_dolares"),
    "Suelo 300 M$": (300e6, "volumen_mediano_dolares"),
    "Suelo 2.000 M$ (elegido)": (2e9, "volumen_mediano_dolares"),
    "Suelo 10.000 M$": (10e9, "volumen_mediano_dolares"),
    "Suelo 2.000 M$ pero con la MEDIA": (2e9, "volumen_medio_dolares"),
}

filas_sensibilidad = []
for nombre, (suelo, columna) in variantes.items():
    fila = {"variante": nombre, "elegibles": (acciones["capitalizacion"] >= suelo).sum()}
    try:
        variante = seleccionar_tickers(suelo, columna)
        fila["coinciden con la elegida"] = len(variante.intersection(seleccion))
        fila["cambian"] = len(variante.difference(seleccion))
    except ValueError:
        fila["coinciden con la elegida"] = None  # no hay 1.000 acciones elegibles: la variante no es viable
        fila["cambian"] = None
    filas_sensibilidad.append(fila)

sensibilidad = pd.DataFrame(filas_sensibilidad).set_index("variante").astype("Int64")  # Int64 admite celdas vacías sin pasar a decimales
sensibilidad

,elegibles,coinciden con la elegida,cambian
variante,,,
Sin suelo (solo acciones con capitalización),2457,992,8
Suelo 300 M$,2446,992,8
Suelo 2.000 M$ (elegido),1945,1000,0
Suelo 10.000 M$,919,<NA>,<NA>
Suelo 2.000 M$ pero con la MEDIA,1945,981,19


¿Qué tickers dejaría entrar quitar el suelo? Son los que el suelo de 2.000 M$ está protegiendo: empresas pequeñas con mucho volumen.

In [20]:
entrarian_sin_suelo = seleccionar_tickers(0).difference(seleccion)
(acciones.loc[entrarian_sin_suelo]
    .join(universo.set_index("ticker")["name"])
    .sort_values("volumen_mediano_dolares", ascending=False)
    .head(15)[["name", "capitalizacion", "volumen_mediano_dolares", "volumen_medio_dolares"]])

,name,capitalizacion,volumen_mediano_dolares,volumen_medio_dolares
ticker,,,,
FCEL,FUELCELL ENERGY INC,1.441574e+09,1.421728e+08,1.539443e+08
KEEL,Keel Infrastructure Corp.,1.883598e+09,1.175747e+08,1.210819e+08
JBLU,JETBLUE AIRWAYS CORP,1.448879e+09,1.097052e+08,1.318435e+08
TE,T1 Energy Inc.,1.015301e+09,1.018736e+08,1.403836e+08
SHAZ,SharonAI Holdings Inc.,1.358027e+09,9.287025e+07,1.044790e+08
EOSE,"Eos Energy Enterprises, Inc.",9.177027e+08,9.160194e+07,9.964244e+07
SEDG,"SOLAREDGE TECHNOLOGIES, INC.",1.981744e+09,9.063261e+07,1.117915e+08
WOLF,"WOLFSPEED, INC.",1.620205e+09,8.792700e+07,1.117982e+08


Y con la media en lugar de la mediana, ¿quién entraría? Lo esperable son acciones con picos puntuales: la media muy por encima de la mediana.

In [21]:
entrarian_con_media = seleccionar_tickers(SUELO_CAPITALIZACION, "volumen_medio_dolares").difference(seleccion)
con_media = acciones.loc[entrarian_con_media].join(universo.set_index("ticker")["name"])
con_media["media / mediana"] = con_media["volumen_medio_dolares"] / con_media["volumen_mediano_dolares"]
print("Ratio media/mediana de toda la selección elegida:",
      round((elegidos["volumen_medio_dolares"] / elegidos["volumen_mediano_dolares"]).median(), 2))
con_media.sort_values("media / mediana", ascending=False).head(15)[["name", "volumen_mediano_dolares", "volumen_medio_dolares", "media / mediana"]]

Ratio media/mediana de toda la selección elegida: 1.11


,name,volumen_mediano_dolares,volumen_medio_dolares,media / mediana
ticker,,,,
KOD,Kodiak Sciences Inc.,2.360912e+07,1.163525e+08,4.928284
VKTX,"Viking Therapeutics, Inc.",6.644640e+07,1.455772e+08,2.190897
OPCH,"Option Care Health, Inc.",4.537606e+07,9.768527e+07,2.152793
PCVX,"Vaxcyte, Inc.",6.339360e+07,1.111857e+08,1.753894
DOCS,"Doximity, Inc.",6.840582e+07,1.143732e+08,1.671981
ITGR,Integer Holdings Corp,8.201425e+07,1.255297e+08,1.530584
BNTX,BioNTech SE,8.307333e+07,1.188645e+08,1.430839
FRVO,Fervo Energy Co,6.961416e+07,9.918514e+07,1.424784
CELC,Celcuity Inc.,8.240258e+07,1.063596e+08,1.290732


## 7. Guardar el universo recortado

Se guardan los 1.000 tickers con su CIK (necesario para la SEC en los notebooks 03 y 06), la capitalización y la liquidez.

In [22]:
universo_recortado = (
    universo.set_index("ticker")
    .loc[seleccion, ["cik", "name", "exchange"]]
    .join(elegidos[["capitalizacion", "volumen_mediano_dolares"]])
    .reset_index()
)
ruta_universo_recortado = CARPETA_CACHE / "universo_recortado.parquet"
universo_recortado.to_parquet(ruta_universo_recortado, index=False)

print("Guardado:", ruta_universo_recortado.name, "-", len(universo_recortado), "filas")
print(universo_recortado["exchange"].value_counts())
universo_recortado.head(10)

Guardado: universo_recortado.parquet - 1000 filas
exchange
NYSE      640
Nasdaq    360
Name: count, dtype: int64


,ticker,cik,name,exchange,capitalizacion,volumen_mediano_dolares
0,MU,0000723125,MICRON TECHNOLOGY INC,Nasdaq,1.164234e+12,2.938481e+10
1,NVDA,0001045810,NVIDIA CORP,Nasdaq,5.536424e+12,2.496771e+10
2,SNDK,0002023554,Sandisk Corp,Nasdaq,2.286166e+11,1.691481e+10
3,AAPL,0000320193,Apple Inc.,Nasdaq,4.912985e+12,1.267465e+10
4,TSLA,0001318605,"Tesla, Inc.",Nasdaq,1.511492e+12,1.179943e+10
5,AMD,0000002488,ADVANCED MICRO DEVICES INC,Nasdaq,9.927080e+11,1.162112e+10
6,MSFT,0000789019,MICROSOFT CORP,Nasdaq,3.973187e+12,1.098514e+10
7,SPCX,0001181412,SPACE EXPLORATION TECHNOLOGIES CORP,Nasdaq,2.142962e+12,1.076495e+10
8,INTC,0000050863,INTEL CORP,Nasdaq,5.534552e+11,1.028815e+10
9,META,0001326801,"Meta Platforms, Inc.",Nasdaq,1.830816e+12,9.847473e+09


## 8. Histórico de 10 años de las 1.000 empresas elegidas

El backtesting necesita hasta 10 años de precios. Ahora que ya sabemos qué 1.000 empresas son, se descarga su OHLCV diario del **10 oct 2016 al 9 oct 2026**, con fechas fijas para que el resultado sea reproducible.

Esta vez los precios van **ajustados por splits y dividendos** (`auto_adjust=True`). Sin ajustar, el día en que la acción deja de incluir el dividendo parece una caída, y el backtesting contaría como pérdida un dinero que el accionista sí cobró. La descarga se guarda en caché: si el fichero ya existe, se lee de disco.

In [23]:
HISTORICO_INICIO = "2016-10-10"
HISTORICO_FIN = "2026-10-10"   # no incluida: el último día es el 9 oct 2026
ruta_historico = CARPETA_CACHE / "precios_historicos.parquet"

if ruta_historico.exists():
    precios_historicos = pd.read_parquet(ruta_historico)
    print("Leído de caché:", ruta_historico.name)
else:
    inicio_descarga = time.time()
    precios_historicos = descargar_precios_en_lotes(universo_recortado["ticker"].tolist(), HISTORICO_INICIO, HISTORICO_FIN, ajustados=True)
    precios_historicos.to_parquet(ruta_historico, index=False)
    print(f"Descargado en {time.time() - inicio_descarga:.0f} s y guardado en {ruta_historico.name}")

precios_historicos = precios_historicos.sort_values(["ticker", "fecha"]).reset_index(drop=True)
print("Filas:", len(precios_historicos), "· Tickers:", precios_historicos["ticker"].nunique())
print("Fechas:", precios_historicos["fecha"].min().date(), "→", precios_historicos["fecha"].max().date())

Leído de caché: precios_historicos.parquet


Filas: 2281091 · Tickers: 1000
Fechas: 2016-10-10 → 2026-10-09


### Por qué ajustar: un dividendo real

Exxon (`XOM`) pagó un dividendo de 1,03 USD con fecha 17 de agosto de 2026. Se compara el retorno de ese día con los precios sin ajustar (la descarga de 6 meses de la sección 3) y con los ajustados:

In [24]:
cierre_sin_ajustar = precios_recientes[precios_recientes["ticker"] == "XOM"].set_index("fecha")["close"]
cierre_ajustado = precios_historicos[precios_historicos["ticker"] == "XOM"].set_index("fecha")["close"]
print(f"XOM, 17 ago 2026 · retorno sin ajustar: {cierre_sin_ajustar.pct_change().loc['2026-08-17']:.2%}"
      f" · ajustado: {cierre_ajustado.pct_change().loc['2026-08-17']:.2%}")

XOM, 17 ago 2026 · retorno sin ajustar: 0.85% · ajustado: 1.50%


Sin ajustar, ese día parece peor de lo que fue para quien tenía la acción: el ajuste devuelve el dividendo al retorno.

### ¿Cuánta historia tiene cada empresa?

In [25]:
anios_de_historia = precios_historicos.groupby("ticker")["fecha"].agg(lambda fechas: (fechas.max() - fechas.min()).days / 365.25)
tramos_historia = pd.cut(anios_de_historia, bins=[0, 1, 3, 5, 9.9, 11], right=False,
                         labels=["< 1 año", "1–3 años", "3–5 años", "5–10 años", "10 años"])
tramos_historia.value_counts().sort_index().rename("empresas")

fecha
< 1 año       14
1–3 años      27
3–5 años      31
5–10 años    129
10 años      799
Name: empresas, dtype: int64

La mayoría tiene los 10 años completos. Las demás salieron a bolsa durante el periodo, así que el principio del backtesting tendrá menos empresas que el final.

### Calidad de los datos

In [26]:
print("Valores vacíos:", precios_historicos.isna().sum().sum())
print("Días repetidos de un ticker:", precios_historicos.duplicated(subset=["fecha", "ticker"]).sum())
print("Máximo por debajo del mínimo:", (precios_historicos["high"] < precios_historicos["low"]).sum())


def dias_fuera_de_rango(columna: str) -> int:
    """Días en que el precio de la columna queda por encima del máximo o por debajo del mínimo (con margen de redondeo)."""
    por_encima = precios_historicos[columna] > precios_historicos["high"] * (1 + 1e-9)
    por_debajo = precios_historicos[columna] < precios_historicos["low"] * (1 - 1e-9)
    return (por_encima | por_debajo).sum()


print("Cierres fuera del rango del día:", dias_fuera_de_rango("close"), "· aperturas fuera del rango:", dias_fuera_de_rango("open"))
print("Días con volumen 0:", (precios_historicos["volume"] == 0).sum(), f"({(precios_historicos['volume'] == 0).mean():.2%})")

Valores vacíos: 0


Días repetidos de un ticker: 0
Máximo por debajo del mínimo: 0
Cierres fuera del rango del día: 0 · aperturas fuera del rango: 3
Días con volumen 0: 6475 (0.28%)


### Saltos extremos

Días con un movimiento de más del 50 % en un solo día:

In [27]:
precios_historicos["retorno"] = precios_historicos.groupby("ticker")["close"].pct_change()
saltos = precios_historicos[precios_historicos["retorno"].abs() > 0.5]
print("Días con saltos de más del 50 %:", len(saltos), "· en", saltos["ticker"].nunique(), "empresas")
saltos.nlargest(5, "retorno")[["fecha", "ticker", "close", "retorno"]]

Días con saltos de más del 50 %: 159 · en 80 empresas


,fecha,ticker,close,retorno
441365,2020-11-20,CHRD,18.844103,257.333377
313676,2025-06-30,BMNR,33.890049,6.948418
22014,2025-07-23,ABVX,68.599998,5.860000
199116,2025-05-07,ASST,67.800003,4.557377
469605,2018-09-19,CLSK,73.000000,3.866667


La mayoría son reales: biotecnológicas con resultados de ensayos y el desplome de 2020. Unos pocos son **cambios de identidad** del valor que Yahoo encadena como una sola serie:

- **CHRD:** quiebra en 2020; las acciones viejas a 0,07 USD y las nuevas a 16,50 USD (+25.733 % en un día).
- **QXO:** antes de diciembre de 2023 era una pequeña sociedad casi vacía.
- **CLSK:** en 2018 negociaba de 10 a 420 acciones al día.

Los datos no se modifican. El notebook 05 evita operar en esas etapas con un filtro de liquidez mínima en la fecha de cada señal.

**Conclusión:** 10 años de precios ajustados de las 1.000 empresas, sin huecos ni días repetidos, con unas pocas anomalías conocidas que se tratan después.

## Comprobaciones

In [28]:
# Precios descargados
assert precios_recientes["fecha"].max() < pd.Timestamp(FECHA_FIN)
assert precios_recientes[["fecha", "ticker", "close"]].notna().all().all()
assert not precios_recientes.duplicated(subset=["fecha", "ticker"]).any(), "Hay días repetidos para un ticker"
assert (precios_recientes["close"] > 0).all()
assert (precios_recientes["volume"] >= 0).all()
assert set(precios_recientes["ticker"]) <= set(universo["ticker"])
assert (liquidez["sesiones"] == VENTANA_VOLUMEN).all()

# Consulta de capitalización: todos los candidatos consultados
assert set(capitalizacion.index) == set(candidatos)

# Universo recortado
assert len(universo_recortado) == NUM_TICKERS
assert universo_recortado["ticker"].is_unique
assert universo_recortado["cik"].is_unique, "Hay empresas con dos clases de acción"
assert (universo_recortado["capitalizacion"] >= SUELO_CAPITALIZACION).all()
assert (capitalizacion.loc[universo_recortado["ticker"], "tipo"] == "EQUITY").all()
assert universo_recortado.notna().all().all()
assert universo_recortado["cik"].str.fullmatch(r"\d{10}").all()

# El prefiltro de volumen no ha cambiado el resultado: el último elegido está por encima
assert universo_recortado["volumen_mediano_dolares"].min() > PREFILTRO_VOLUMEN

# Histórico de 10 años: las mismas 1.000 empresas, sin vacíos ni días repetidos, y precios coherentes
assert set(precios_historicos["ticker"]) == set(universo_recortado["ticker"])
assert precios_historicos[["fecha", "ticker", "open", "high", "low", "close", "volume"]].notna().all().all()
assert not precios_historicos.duplicated(subset=["fecha", "ticker"]).any()
assert (precios_historicos["high"] >= precios_historicos["low"]).all()
assert precios_historicos["fecha"].min() >= pd.Timestamp(HISTORICO_INICIO)
assert precios_historicos["fecha"].max() < pd.Timestamp(HISTORICO_FIN)

print("Todas las comprobaciones OK")

Todas las comprobaciones OK
